# Demo 07 : Testing agents

Run the cells in order and watch. There is nothing to write.

## a · scripted-fake-model

We script tool calls with `GenericFakeChatModel`. Agents with tools need a small subclass that overrides `bind_tools`. These tests exercise the tool and graph wiring. They do not test the real model's choices.

In [1]:
import shutil
import urllib.request

from pathlib import Path
here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / "data/game-studio/studio.db").exists()), None)
DB = here / "studio.db"
if root:
    shutil.copyfile(root / "data/game-studio/studio.db", DB)
elif not DB.exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/data/game-studio/studio.db", DB)

In [2]:
from contextlib import closing
import sqlite3
from pathlib import Path
from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, ToolCall
from langchain.tools import tool
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langgraph.checkpoint.memory import InMemorySaver


class FakeToolModel(GenericFakeChatModel):
    def bind_tools(self, tools, **kwargs):
        return self


def asset_tool(db):
    @tool
    def asset_status(asset_id: str) -> str:
        """Read the status of a Comet Arcade asset, draft or final."""
        with closing(sqlite3.connect(db)) as conn:
            row = conn.execute("SELECT status FROM assets WHERE asset_id = ?", (asset_id,)).fetchone()
        return str(row[0]) if row else "Unknown asset"
    return asset_status


def scripted(db, *messages):
    return create_agent(FakeToolModel(messages=iter(messages)), tools=[asset_tool(db)],
                        checkpointer=InMemorySaver())


def call(asset_id="CS-SPR-03", call_id="asset-1"):
    return AIMessage("", tool_calls=[ToolCall(name="asset_status", args={"asset_id": asset_id}, id=call_id)])

In [3]:
try:
    plain = GenericFakeChatModel(messages=iter([AIMessage("Hello.")]))
    create_agent(plain, tools=[asset_tool(DB)]).invoke({"messages": [HumanMessage("Status?")]})
except NotImplementedError:
    print("plain fake: bind_tools is not implemented")
agent = scripted(DB, call(), AIMessage("CS-SPR-03 is still a draft."))
config = {"configurable": {"thread_id": "asset-demo"}}
out = agent.invoke({"messages": [HumanMessage("Status of CS-SPR-03?")]}, config)
for message in out["messages"]:
    print(message.type, message.content or getattr(message, "tool_calls", []))
print("saved messages:", len(agent.get_state(config).values["messages"]))

plain fake: bind_tools is not implemented
human Status of CS-SPR-03?
ai [{'name': 'asset_status', 'args': {'asset_id': 'CS-SPR-03'}, 'id': 'asset-1', 'type': 'tool_call'}]
tool draft
ai CS-SPR-03 is still a draft.
saved messages: 4


## b · pytest

We write a test file next to the notebook and run pytest in a subprocess. We assert the real tool result, its call identifier, the saved history and thread isolation. The final answer is scripted and cannot prove answer quality.

In [4]:
from pathlib import Path
import subprocess
import sys

test_source = '\nfrom contextlib import closing\nimport sqlite3\nfrom pathlib import Path\nfrom langchain.agents import create_agent\nfrom langchain.messages import AIMessage, HumanMessage, ToolCall\nfrom langchain.tools import tool\nfrom langchain_core.language_models.fake_chat_models import GenericFakeChatModel\nfrom langgraph.checkpoint.memory import InMemorySaver\n\n\nclass FakeToolModel(GenericFakeChatModel):\n    def bind_tools(self, tools, **kwargs):\n        return self\n\n\ndef asset_tool(db):\n    @tool\n    def asset_status(asset_id: str) -> str:\n        """Read the status of a Comet Arcade asset, draft or final."""\n        with closing(sqlite3.connect(db)) as conn:\n            row = conn.execute("SELECT status FROM assets WHERE asset_id = ?", (asset_id,)).fetchone()\n        return str(row[0]) if row else "Unknown asset"\n    return asset_status\n\n\ndef scripted(db, *messages):\n    return create_agent(FakeToolModel(messages=iter(messages)), tools=[asset_tool(db)],\n                        checkpointer=InMemorySaver())\n\n\ndef call(asset_id="CS-SPR-03", call_id="asset-1"):\n    return AIMessage("", tool_calls=[ToolCall(name="asset_status", args={"asset_id": asset_id}, id=call_id)])\n\nimport pytest\n\n\n@pytest.fixture\ndef db(tmp_path):\n    import shutil\n    import urllib.request\n\n    here = Path(__file__).resolve().parent\n    root = next((p for p in [here, *here.parents] if (p / "data/game-studio/studio.db").exists()), None)\n    target = tmp_path / "studio.db"\n    if root:\n        shutil.copyfile(root / "data/game-studio/studio.db", target)\n    else:\n        urllib.request.urlretrieve("https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/data/game-studio/studio.db", target)\n    return target\n\n\ndef test_tool_result_and_call_id(db):\n    agent = scripted(db, call(), AIMessage("CS-SPR-03 is a draft."))\n    out = agent.invoke({"messages": [HumanMessage("Status of CS-SPR-03?")]},\n                       {"configurable": {"thread_id": "asset"}})\n    results = [m for m in out["messages"] if m.type == "tool"]\n    assert len(results) == 1\n    assert results[0].content == "draft"\n    assert results[0].tool_call_id == "asset-1"\n\n\ndef test_unknown_asset_is_explicit(db):\n    assert asset_tool(db).invoke({"asset_id": "UNKNOWN"}) == "Unknown asset"\n\n\ndef test_memory_and_thread_isolation(db):\n    agent = scripted(db, call(), AIMessage("A draft."), AIMessage("CS-SPR-03."), AIMessage("No earlier lookup."))\n    config = {"configurable": {"thread_id": "first"}}\n    agent.invoke({"messages": [HumanMessage("Status?")]}, config)\n    agent.invoke({"messages": [HumanMessage("Which asset?")]}, config)\n    saved = agent.get_state(config).values["messages"]\n    assert any(m.type == "tool" and m.content == "draft" for m in saved)\n    other = {"configurable": {"thread_id": "other"}}\n    out = agent.invoke({"messages": [HumanMessage("Which asset?")]}, other)\n    assert [m.type for m in out["messages"]] == ["human", "ai"]\n    assert len(agent.get_state(config).values["messages"]) == len(saved)\n'
test_path = Path("test_module04_agent.py")
test_path.write_text(test_source, encoding="utf-8")
try:
    run = subprocess.run([sys.executable, "-B", "-m", "pytest", "-p", "no:cacheprovider", "-q", str(test_path)],
                         capture_output=True, text=True)
    print(run.stdout)
    if run.stderr:
        print(run.stderr)
    run.check_returncode()
finally:
    DB.unlink(missing_ok=True)

...                                                                      [100%]
3 passed in 0.75s



## Things to try

Optional : we can change the game, the asset or the question and run the example again.